# YOLOv8 Model Comparison Notebook

This notebook compares two YOLOv8 model files:
- `updated-model.pt`
- `best.pt`

We'll analyze their architecture, performance metrics, and differences.


In [1]:
import sys
print("Python executable:", sys.executable)
print("Virtual env active:", '/home/ubuntu/obc-yolov8/.venv' in sys.executable)

Python executable: /home/ubuntu/obc-yolov8/.venv/bin/python
Virtual env active: True


In [2]:
# Import required libraries
from ultralytics import YOLO
import torch
import numpy as np
import pandas as pd
import os
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Set up paths
model1_path = "/home/ubuntu/obc-yolov8/OBC-YOLOv8/ultralytics10.24/updated-model.pt"
model2_path = "/home/ubuntu/obc-yolov8/OBC-YOLOv8/ultralytics10.24/best.pt"

print("Model paths:")
print(f"Model 1: {model1_path}")
print(f"Model 2: {model2_path}")
print(f"Model 1 exists: {os.path.exists(model1_path)}")
print(f"Model 2 exists: {os.path.exists(model2_path)}")


/home/ubuntu/obc-yolov8/.venv/lib/python3.10/site-packages/mmcv/__init__.py:20: UserWarning: On January 1, 2023, MMCV will release v2.0.0, in which it will remove components related to the training process and add a data transformation module. In addition, it will rename the package names mmcv to mmcv-lite and mmcv-full to mmcv. See https://github.com/open-mmlab/mmcv/blob/master/docs/en/compatibility.md for more details.
  warnings.warn(
/home/ubuntu/obc-yolov8/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/ubuntu/obc-yolov8/.venv/lib/python3.10/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


Model paths:
Model 1: /home/ubuntu/obc-yolov8/OBC-YOLOv8/ultralytics10.24/updated-model.pt
Model 2: /home/ubuntu/obc-yolov8/OBC-YOLOv8/ultralytics10.24/best.pt
Model 1 exists: True
Model 2 exists: True


## 1. Load and Basic Information


In [3]:
def load_model_info(model_path, model_name):
    """Load model and extract basic information"""
    print(f"\n{'='*50}")
    print(f"Loading {model_name}: {model_path}")
    print(f"{'='*50}")
    
    try:
        # Load model
        model = YOLO(model_path)
        
        # Basic info
        print(f"✅ Model loaded successfully")
        print(f"📁 Model Type: {model.model.__class__.__name__}")
        print(f"🏷️  Classes: {list(model.names.values())}")
        print(f"🔢 Number of Classes: {len(model.names)}")
        
        # Model size
        model_size_mb = os.path.getsize(model_path) / (1024 * 1024)
        print(f"💾 Model Size: {model_size_mb:.2f} MB")
        
        # Parameters
        total_params = sum(p.numel() for p in model.model.parameters())
        trainable_params = sum(p.numel() for p in model.model.parameters() if p.requires_grad)
        print(f"🔢 Total Parameters: {total_params:,}")
        print(f"🔢 Trainable Parameters: {trainable_params:,}")
        
        # Device info
        print(f"🖥️  Device: {model.device}")
        
        return {
            'model': model,
            'model_size_mb': model_size_mb,
            'total_params': total_params,
            'trainable_params': trainable_params,
            'num_classes': len(model.names),
            'class_names': list(model.names.values())
        }
        
    except Exception as e:
        print(f"❌ Error loading model: {e}")
        return None

# Load both models
model1_info = load_model_info(model1_path, "Updated Model")
model2_info = load_model_info(model2_path, "Best Model")



Loading Updated Model: /home/ubuntu/obc-yolov8/OBC-YOLOv8/ultralytics10.24/updated-model.pt
✅ Model loaded successfully
📁 Model Type: DetectionModel
🏷️  Classes: ['person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus', 'train', 'truck', 'boat', 'traffic light', 'fire hydrant', 'stop sign', 'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse', 'sheep', 'cow', 'elephant', 'bear', 'zebra', 'giraffe', 'backpack', 'umbrella', 'handbag', 'tie', 'suitcase', 'frisbee', 'skis', 'snowboard', 'sports ball', 'kite', 'baseball bat', 'baseball glove', 'skateboard', 'surfboard', 'tennis racket', 'bottle', 'wine glass', 'cup', 'fork', 'knife', 'spoon', 'bowl', 'banana', 'apple', 'sandwich', 'orange', 'broccoli', 'carrot', 'hot dog', 'pizza', 'donut', 'cake', 'chair', 'couch', 'potted plant', 'bed', 'dining table', 'toilet', 'tv', 'laptop', 'mouse', 'remote', 'keyboard', 'cell phone', 'microwave', 'oven', 'toaster', 'sink', 'refrigerator', 'book', 'clock', 'vase', 'scissors', 'teddy bear', 'ha

## 2. Architecture Comparison


In [4]:
def compare_architectures(model1_info, model2_info):
    """Compare model architectures"""
    print("\n" + "="*60)
    print("🏗️  ARCHITECTURE COMPARISON")
    print("="*60)
    
    if model1_info and model2_info:
        # Create comparison table
        comparison_data = {
            'Metric': [
                'Model Size (MB)',
                'Total Parameters',
                'Trainable Parameters',
                'Number of Classes',
                'Device'
            ],
            'Updated Model': [
                f"{model1_info['model_size_mb']:.2f}",
                f"{model1_info['total_params']:,}",
                f"{model1_info['trainable_params']:,}",
                model1_info['num_classes'],
                str(model1_info['model'].device)
            ],
            'Best Model': [
                f"{model2_info['model_size_mb']:.2f}",
                f"{model2_info['total_params']:,}",
                f"{model2_info['trainable_params']:,}",
                model2_info['num_classes'],
                str(model2_info['model'].device)
            ]
        }
        
        df_comparison = pd.DataFrame(comparison_data)
        print(df_comparison.to_string(index=False))
        
        # Check for differences
        print("\n🔍 Differences:")
        if model1_info['total_params'] != model2_info['total_params']:
            diff = model1_info['total_params'] - model2_info['total_params']
            print(f"  - Parameter difference: {diff:,} ({diff/model2_info['total_params']*100:+.1f}%)")
        
        if model1_info['model_size_mb'] != model2_info['model_size_mb']:
            diff = model1_info['model_size_mb'] - model2_info['model_size_mb']
            print(f"  - Size difference: {diff:.2f} MB ({diff/model2_info['model_size_mb']*100:+.1f}%)")
        
        if model1_info['num_classes'] != model2_info['num_classes']:
            print(f"  - Class count difference: {model1_info['num_classes']} vs {model2_info['num_classes']}")
        
        if model1_info['class_names'] != model2_info['class_names']:
            print(f"  - Class names difference detected")
            print(f"    Updated Model: {model1_info['class_names']}")
            print(f"    Best Model: {model2_info['class_names']}")
    else:
        print("❌ Cannot compare - one or both models failed to load")

compare_architectures(model1_info, model2_info)



🏗️  ARCHITECTURE COMPARISON
              Metric Updated Model Best Model
     Model Size (MB)          6.24       6.24
    Total Parameters     3,157,200  3,157,200
Trainable Parameters             0          0
   Number of Classes            80         80
              Device           cpu        cpu

🔍 Differences:
  - Size difference: 0.00 MB (+0.0%)


## 3. Performance Metrics Comparison


In [5]:
def get_model_metrics(model_info, model_name):
    """Get performance metrics for a model"""
    if not model_info:
        return None
    
    print(f"\n📊 Getting metrics for {model_name}...")
    
    try:
        model = model_info['model']
        
        # Run validation
        print("  Running validation...")
        results = model.val(plots=False, save=False, verbose=False)
        
        # Extract metrics
        metrics = {}
        
        if hasattr(results, 'box'):
            box_metrics = results.box
            metrics['map50'] = box_metrics.map50
            metrics['map50_95'] = box_metrics.map
            metrics['precision'] = box_metrics.mp
            metrics['recall'] = box_metrics.mr
            
            # Calculate F1-score
            if metrics['precision'] + metrics['recall'] > 0:
                metrics['f1_score'] = 2 * (metrics['precision'] * metrics['recall']) / (metrics['precision'] + metrics['recall'])
            else:
                metrics['f1_score'] = 0
        
        # Speed metrics
        if hasattr(results, 'speed'):
            speed = results.speed
            metrics['inference_time'] = speed['inference']
            metrics['total_time'] = sum(speed.values())
        
        # Add architecture info
        metrics['total_params'] = model_info['total_params']
        metrics['model_size_mb'] = model_info['model_size_mb']
        
        print(f"  ✅ Metrics extracted successfully")
        return metrics
        
    except Exception as e:
        print(f"  ❌ Error getting metrics: {e}")
        return None

# Get metrics for both models
metrics1 = get_model_metrics(model1_info, "Updated Model")
metrics2 = get_model_metrics(model2_info, "Best Model")


Ultralytics YOLOv8.0.164 🚀 Python-3.10.9 torch-2.5.1+cu121 CUDA:0 (Tesla T4, 15102MiB)



📊 Getting metrics for Updated Model...
  Running validation...


YOLOv8 summary (fused): 168 layers, 3151904 parameters, 0 gradients, 8.7 GFLOPs
Ultralytics YOLOv8.0.164 🚀 Python-3.10.9 torch-2.5.1+cu121 CUDA:0 (Tesla T4, 15102MiB)
YOLOv8 summary (fused): 168 layers, 3151904 parameters, 0 gradients, 8.7 GFLOPs


  ❌ Error getting metrics: [Errno 13] Permission denied: '/root/ultralytics/ultralytics/cfg/datasets/coco128.yaml'

📊 Getting metrics for Best Model...
  Running validation...
  ❌ Error getting metrics: [Errno 13] Permission denied: '/root/ultralytics/ultralytics/cfg/datasets/coco128.yaml'


In [6]:
def compare_metrics(metrics1, metrics2):
    """Compare performance metrics between models"""
    print("\n" + "="*60)
    print("📈 PERFORMANCE METRICS COMPARISON")
    print("="*60)
    
    if metrics1 and metrics2:
        # Create comparison table
        comparison_data = {
            'Metric': [
                'mAP@0.5',
                'mAP@0.5:0.95',
                'Precision',
                'Recall',
                'F1-Score',
                'Inference Time (ms)',
                'Total Time (ms)',
                'Parameters',
                'Model Size (MB)'
            ],
            'Updated Model': [
                f"{metrics1.get('map50', 0):.3f}",
                f"{metrics1.get('map50_95', 0):.3f}",
                f"{metrics1.get('precision', 0):.3f}",
                f"{metrics1.get('recall', 0):.3f}",
                f"{metrics1.get('f1_score', 0):.3f}",
                f"{metrics1.get('inference_time', 0):.1f}",
                f"{metrics1.get('total_time', 0):.1f}",
                f"{metrics1.get('total_params', 0):,}",
                f"{metrics1.get('model_size_mb', 0):.2f}"
            ],
            'Best Model': [
                f"{metrics2.get('map50', 0):.3f}",
                f"{metrics2.get('map50_95', 0):.3f}",
                f"{metrics2.get('precision', 0):.3f}",
                f"{metrics2.get('recall', 0):.3f}",
                f"{metrics2.get('f1_score', 0):.3f}",
                f"{metrics2.get('inference_time', 0):.1f}",
                f"{metrics2.get('total_time', 0):.1f}",
                f"{metrics2.get('total_params', 0):,}",
                f"{metrics2.get('model_size_mb', 0):.2f}"
            ]
        }
        
        df_metrics = pd.DataFrame(comparison_data)
        print(df_metrics.to_string(index=False))
        
        # Calculate improvements
        print("\n📊 Performance Improvements (Updated vs Best):")
        
        key_metrics = ['map50', 'map50_95', 'precision', 'recall', 'f1_score']
        for metric in key_metrics:
            if metric in metrics1 and metric in metrics2:
                val1 = metrics1[metric]
                val2 = metrics2[metric]
                if val2 > 0:
                    improvement = ((val1 - val2) / val2) * 100
                    direction = "📈" if improvement > 0 else "📉" if improvement < 0 else "➡️"
                    print(f"  {direction} {metric.upper()}: {improvement:+.2f}% ({val1:.3f} vs {val2:.3f})")
        
        # Speed comparison
        if 'inference_time' in metrics1 and 'inference_time' in metrics2:
            speed_diff = metrics1['inference_time'] - metrics2['inference_time']
            speed_improvement = (speed_diff / metrics2['inference_time']) * 100 if metrics2['inference_time'] > 0 else 0
            direction = "⚡" if speed_improvement < 0 else "🐌" if speed_improvement > 0 else "➡️"
            print(f"  {direction} Inference Speed: {speed_improvement:+.1f}% ({metrics1['inference_time']:.1f}ms vs {metrics2['inference_time']:.1f}ms)")
    else:
        print("❌ Cannot compare metrics - one or both models failed to load or validate")

compare_metrics(metrics1, metrics2)



📈 PERFORMANCE METRICS COMPARISON
❌ Cannot compare metrics - one or both models failed to load or validate


## 4. Visual Comparison


In [ ]:
def create_comparison_plots(metrics1, metrics2):
    """Create visual comparison plots"""
    if not metrics1 or not metrics2:
        print("❌ Cannot create plots - missing metrics data")
        return
    
    # Set up the plotting style
    plt.style.use('default')
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle('YOLOv8 Model Comparison', fontsize=16, fontweight='bold')
    
    # 1. Performance Metrics Comparison
    ax1 = axes[0, 0]
    performance_metrics = ['map50', 'map50_95', 'precision', 'recall', 'f1_score']
    updated_values = [metrics1.get(m, 0) for m in performance_metrics]
    best_values = [metrics2.get(m, 0) for m in performance_metrics]
    
    x = np.arange(len(performance_metrics))
    width = 0.35
    
    ax1.bar(x - width/2, updated_values, width, label='Updated Model', alpha=0.8, color='skyblue')
    ax1.bar(x + width/2, best_values, width, label='Best Model', alpha=0.8, color='lightcoral')
    
    ax1.set_xlabel('Metrics')
    ax1.set_ylabel('Score')
    ax1.set_title('Performance Metrics Comparison')
    ax1.set_xticks(x)
    ax1.set_xticklabels([m.upper() for m in performance_metrics], rotation=45)
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # 2. Model Size and Parameters
    ax2 = axes[0, 1]
    size_metrics = ['Model Size (MB)', 'Parameters (M)']
    updated_size = [metrics1.get('model_size_mb', 0), metrics1.get('total_params', 0)/1e6]
    best_size = [metrics2.get('model_size_mb', 0), metrics2.get('total_params', 0)/1e6]
    
    x2 = np.arange(len(size_metrics))
    ax2.bar(x2 - width/2, updated_size, width, label='Updated Model', alpha=0.8, color='lightgreen')
    ax2.bar(x2 + width/2, best_size, width, label='Best Model', alpha=0.8, color='orange')
    
    ax2.set_xlabel('Metrics')
    ax2.set_ylabel('Value')
    ax2.set_title('Model Size & Parameters')
    ax2.set_xticks(x2)
    ax2.set_xticklabels(size_metrics)
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # 3. Speed Comparison
    ax3 = axes[1, 0]
    speed_metrics = ['Inference Time (ms)', 'Total Time (ms)']
    updated_speed = [metrics1.get('inference_time', 0), metrics1.get('total_time', 0)]
    best_speed = [metrics2.get('inference_time', 0), metrics2.get('total_time', 0)]
    
    x3 = np.arange(len(speed_metrics))
    ax3.bar(x3 - width/2, updated_speed, width, label='Updated Model', alpha=0.8, color='gold')
    ax3.bar(x3 + width/2, best_speed, width, label='Best Model', alpha=0.8, color='purple')
    
    ax3.set_xlabel('Metrics')
    ax3.set_ylabel('Time (ms)')
    ax3.set_title('Speed Comparison')
    ax3.set_xticks(x3)
    ax3.set_xticklabels(speed_metrics)
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    
    # 4. Improvement Summary
    ax4 = axes[1, 1]
    improvements = []
    metric_names = []
    
    for metric in performance_metrics:
        if metric in metrics1 and metric in metrics2:
            val1 = metrics1[metric]
            val2 = metrics2[metric]
            if val2 > 0:
                improvement = ((val1 - val2) / val2) * 100
                improvements.append(improvement)
                metric_names.append(metric.upper())
    
    colors = ['green' if x > 0 else 'red' for x in improvements]
    ax4.bar(metric_names, improvements, color=colors, alpha=0.7)
    ax4.set_xlabel('Metrics')
    ax4.set_ylabel('Improvement (%)')
    ax4.set_title('Performance Improvement (Updated vs Best)')
    ax4.axhline(y=0, color='black', linestyle='-', alpha=0.3)
    ax4.grid(True, alpha=0.3)
    plt.setp(ax4.get_xticklabels(), rotation=45)
    
    plt.tight_layout()
    plt.show()
    
    print("📊 Comparison plots generated successfully!")

create_comparison_plots(metrics1, metrics2)


## 5. Export Results and Summary


In [ ]:
def export_comparison_results(metrics1, metrics2):
    """Export comparison results to CSV"""
    if not metrics1 or not metrics2:
        print("❌ Cannot export - missing metrics data")
        return None
    
    # Prepare data for export
    export_data = {
        'Model': ['Updated Model', 'Best Model'],
        'mAP@0.5': [metrics1.get('map50', 0), metrics2.get('map50', 0)],
        'mAP@0.5:0.95': [metrics1.get('map50_95', 0), metrics2.get('map50_95', 0)],
        'Precision': [metrics1.get('precision', 0), metrics2.get('precision', 0)],
        'Recall': [metrics1.get('recall', 0), metrics2.get('recall', 0)],
        'F1-Score': [metrics1.get('f1_score', 0), metrics2.get('f1_score', 0)],
        'Inference_Time_ms': [metrics1.get('inference_time', 0), metrics2.get('inference_time', 0)],
        'Total_Time_ms': [metrics1.get('total_time', 0), metrics2.get('total_time', 0)],
        'Parameters': [metrics1.get('total_params', 0), metrics2.get('total_params', 0)],
        'Model_Size_MB': [metrics1.get('model_size_mb', 0), metrics2.get('model_size_mb', 0)]
    }
    
    # Create DataFrame
    df_export = pd.DataFrame(export_data)
    
    # Export to CSV
    csv_path = "/home/ubuntu/obc-yolov8/OBC-YOLOv8/ultralytics10.24/model_comparison_results.csv"
    df_export.to_csv(csv_path, index=False)
    
    print(f"\n💾 Results exported to: {csv_path}")
    print("\n📋 Exported Data:")
    print(df_export.to_string(index=False))
    
    return df_export

def generate_summary(metrics1, metrics2):
    """Generate a summary and recommendations"""
    print("\n" + "="*60)
    print("📋 SUMMARY AND RECOMMENDATIONS")
    print("="*60)
    
    if not metrics1 or not metrics2:
        print("❌ Cannot generate summary - missing metrics data")
        return
    
    # Determine which model is better
    map50_1 = metrics1.get('map50', 0)
    map50_2 = metrics2.get('map50', 0)
    
    print(f"\n🎯 Key Findings:")
    print(f"  • Updated Model mAP@0.5: {map50_1:.3f} ({map50_1*100:.1f}%)")
    print(f"  • Best Model mAP@0.5: {map50_2:.3f} ({map50_2*100:.1f}%)")
    
    if map50_1 > map50_2:
        improvement = ((map50_1 - map50_2) / map50_2) * 100
        print(f"  • Updated Model is BETTER by {improvement:.1f}%")
        better_model = "Updated Model"
    elif map50_2 > map50_1:
        improvement = ((map50_2 - map50_1) / map50_1) * 100
        print(f"  • Best Model is BETTER by {improvement:.1f}%")
        better_model = "Best Model"
    else:
        print(f"  • Both models have similar performance")
        better_model = "Similar"
    
    print(f"\n💡 Recommendations:")
    
    if better_model == "Updated Model":
        print(f"  ✅ Use the Updated Model for production")
        print(f"  📈 It shows improved performance over the Best Model")
    elif better_model == "Best Model":
        print(f"  ✅ Use the Best Model for production")
        print(f"  📈 It shows better performance than the Updated Model")
    else:
        print(f"  ⚖️  Both models perform similarly")
        print(f"  🔍 Consider other factors like model size, speed, or training time")
    
    # Additional considerations
    size1 = metrics1.get('model_size_mb', 0)
    size2 = metrics2.get('model_size_mb', 0)
    
    if abs(size1 - size2) > 1.0:  # Significant size difference
        if size1 < size2:
            print(f"  💾 Updated Model is smaller ({size1:.1f}MB vs {size2:.1f}MB) - better for deployment")
        else:
            print(f"  💾 Best Model is smaller ({size2:.1f}MB vs {size1:.1f}MB) - better for deployment")
    
    speed1 = metrics1.get('inference_time', 0)
    speed2 = metrics2.get('inference_time', 0)
    
    if abs(speed1 - speed2) > 1.0:  # Significant speed difference
        if speed1 < speed2:
            print(f"  ⚡ Updated Model is faster ({speed1:.1f}ms vs {speed2:.1f}ms) - better for real-time applications")
        else:
            print(f"  ⚡ Best Model is faster ({speed2:.1f}ms vs {speed1:.1f}ms) - better for real-time applications")
    
    print(f"\n🔬 Next Steps:")
    print(f"  • Test both models on your specific use case")
    print(f"  • Consider ensemble methods if both models have strengths")
    print(f"  • Monitor performance on new data")
    print(f"  • Consider fine-tuning the better model further")

# Export results and generate summary
export_df = export_comparison_results(metrics1, metrics2)
generate_summary(metrics1, metrics2)


---

## 🎉 Analysis Complete!

This notebook has provided a comprehensive comparison of your two YOLOv8 model files:
- `updated-model.pt`
- `best.pt`

The analysis includes:
- ✅ Model architecture comparison
- ✅ Performance metrics evaluation
- ✅ Visual comparison charts
- ✅ Export of results to CSV
- ✅ Summary and recommendations

Use the results to make informed decisions about which model to use for your specific application!
